# Bronze ingestion — MDS (asset registry)

This notebook takes the raw `mds` table, keeps a chosen set of columns, runs a quick
profile, and saves the result as `hive_metastore.bronze.bronze_mds`.

Unlike the pure-copy bronze notebooks, this one keeps only 13 columns and drops the rest,
so `bronze_mds` is a trimmed-down version of the source rather than a full copy.

From the columns this looks like an asset / installation registry — one of its fields,
`CONCELHO` (municipality), is used as a model feature later, and `X_SIT` / `Y_SIT` look
like location coordinates.

## 1. Setup

Load the shared helper functions and imports from the project utilities notebook.

In [0]:
%run /Workspace/Users/daniel.branco@cgi.com/Tese/00_Utils

## 2. Load and select columns

Read the source table, then keep the 13 columns of interest and drop the rest. The
preview and schema print-out confirm the source loaded correctly before selecting.

In [0]:
df = spark.read.table("hive_metastore.default.mds")
display(df.limit(5))
df.printSchema()

Keep the chosen columns:
`ID_MDS`, `ID_OBJECTO`, `CONCELHO`, `AO`, `NOME`, `TIPOINST`, `TAGCOM`, `REDE`,
`CHAVE_SIT`, `X_SIT`, `Y_SIT`, `MARCA`, `MODELO`.

In [0]:
mds = df.select("ID_MDS","ID_OBJECTO","CONCELHO","AO","NOME","TIPOINST","TAGCOM","REDE","CHAVE_SIT","X_SIT","Y_SIT","MARCA","MODELO")

## 3. Profile

Use the built-in `summarize` to get per-column stats (counts, distinct values, missing
values, distributions) for the selected columns. This is the profiling step for this
table, in place of the manual null/zero checks used in the other bronze notebooks.

In [0]:
dbutils.data.summarize(mds)

## 4. Save to bronze

Save the selected columns as Delta. `overwrite` plus `overwriteSchema` means you can
re-run this cell safely.

**Target table:** `hive_metastore.bronze.bronze_mds`

In [0]:
target_catalog = "hive_metastore"     # change this
target_schema = "bronze"
target_table = "bronze_mds"  # change this

full_name = f"{target_catalog}.{target_schema}.{target_table}"


Save the table and show the result.

In [0]:
(
    mds.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(full_name)
)

display(spark.table(full_name).limit(20))
